# 🌿 Crop Disease Detection — Training Notebook

This notebook downloads the Plant Disease Prediction dataset from Kaggle,
trains a MobileNetV2 transfer-learning model, evaluates it, and saves all
artifacts to Google Drive.

**Run every cell in order.** If Colab disconnects, re-run from Cell 1 —
the dataset will not be re-downloaded and training will resume from the
latest checkpoint automatically.

## 1 · Setup Environment

In [ ]:
!pip install -q kaggle scikit-learn seaborn

import os, json, shutil, zipfile, glob
from collections import Counter

from google.colab import drive, files
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Mount Drive
drive.mount('/content/drive')

DRIVE_PATH    = '/content/drive/MyDrive/PlantDiseaseDataset'
ARTIFACTS_DIR = os.path.join(DRIVE_PATH, 'artifacts')
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# Report hardware
gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow {tf.__version__}  —  GPUs detected: {len(gpus)}')
for g in gpus:
    print(f'  • {g.name}')

## 2 · Kaggle API Setup

Upload your `kaggle.json` token.  The file is moved to `~/.kaggle/` and
any copy left in the working directory or Drive is deleted immediately.

In [ ]:
KAGGLE_DIR = os.path.expanduser('~/.kaggle')
KAGGLE_JSON = os.path.join(KAGGLE_DIR, 'kaggle.json')

if not os.path.exists(KAGGLE_JSON):
    print('Please upload your kaggle.json file:')
    uploaded = files.upload()
    if 'kaggle.json' not in uploaded:
        raise RuntimeError('kaggle.json was not uploaded. Re-run this cell.')
    os.makedirs(KAGGLE_DIR, exist_ok=True)
    shutil.move('kaggle.json', KAGGLE_JSON)
    os.chmod(KAGGLE_JSON, 0o600)
    print('Kaggle API configured.')
else:
    print('Kaggle API already configured.')

# Remove any stray copies in the Drive folder
for stray in glob.glob(os.path.join(DRIVE_PATH, 'kaggle.json')):
    os.remove(stray)
    print(f'Removed stray credentials: {stray}')

## 3 · Download & Extract Dataset

In [ ]:
DATASET_SLUG = 'shuvranshu/plant-disease-prediction-dataset'
ZIP_PATH     = os.path.join(DRIVE_PATH, 'plant-disease-prediction-dataset.zip')
EXTRACT_DIR  = os.path.join(DRIVE_PATH, 'extracted_data')

if not os.path.exists(ZIP_PATH):
    print('Downloading dataset …')
    !kaggle datasets download -d {DATASET_SLUG} -p {DRIVE_PATH}
    if not os.path.exists(ZIP_PATH):
        raise FileNotFoundError(
            'Download failed — check your Kaggle token and internet connection.')
    print('Download complete.')
else:
    print('Dataset ZIP already exists.')

if not os.path.isdir(EXTRACT_DIR):
    print('Extracting …')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    print('Extraction complete.')
else:
    print('Dataset already extracted.')

## 4 · Inspect Dataset Structure

Automatically find the directory whose subdirectories are the class labels.
Handles nested folders, official train/valid splits, and empty directories.

In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png'}

def is_image(fname):
    return os.path.splitext(fname)[1].lower() in IMAGE_EXTS

def class_subdirs(directory):
    """Return sorted subdirectory names that contain ≥ 1 image."""
    result = []
    for name in sorted(os.listdir(directory)):
        sub = os.path.join(directory, name)
        if os.path.isdir(sub) and any(is_image(f) for f in os.listdir(sub)):
            result.append(name)
    return result

def find_class_dir(base):
    """Walk the tree and find the class-label directory."""
    if not os.path.isdir(base):
        raise FileNotFoundError(f'Directory not found: {base}')
    # Pass 1: look for an official train/training split
    for root, dirs, _ in os.walk(base):
        for d in sorted(dirs):
            if d.lower() in ('train', 'training'):
                candidate = os.path.join(root, d)
                if len(class_subdirs(candidate)) >= 2:
                    return candidate
    # Pass 2: shallowest dir with ≥ 2 image-containing subdirs
    for root, dirs, _ in sorted(os.walk(base), key=lambda t: t[0].count(os.sep)):
        if len(class_subdirs(root)) >= 2:
            return root
    raise ValueError(f'No valid class directory found under {base}')

TARGET_DIR  = find_class_dir(EXTRACT_DIR)
CLASS_NAMES = class_subdirs(TARGET_DIR)
NUM_CLASSES = len(CLASS_NAMES)

print(f'Class directory : {TARGET_DIR}')
print(f'Classes found   : {NUM_CLASSES}')

total = 0
for cls in CLASS_NAMES:
    n = sum(1 for f in os.listdir(os.path.join(TARGET_DIR, cls)) if is_image(f))
    print(f'  {cls:40s} {n:>6d} images')
    total += n
print(f'  {"TOTAL":40s} {total:>6d} images')

if NUM_CLASSES < 2:
    raise ValueError('Need at least 2 classes to train a classifier.')

## 5 · Stratified Data Splits

Build disjoint 80 / 10 / 10 splits from the file paths.  Stratification
is used when every class has enough samples; otherwise falls back to
random splitting to avoid crashes.

In [ ]:
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
SEED       = 42

# Collect all (filepath, label_index) pairs
class_to_idx = {c: i for i, c in enumerate(CLASS_NAMES)}
all_paths, all_labels = [], []
for cls in CLASS_NAMES:
    cls_dir = os.path.join(TARGET_DIR, cls)
    for f in os.listdir(cls_dir):
        if is_image(f):
            all_paths.append(os.path.join(cls_dir, f))
            all_labels.append(class_to_idx[cls])

print(f'Total images collected: {len(all_paths)}')

# Stratification safety
counts = Counter(all_labels)
can_strat = all(c >= 3 for c in counts.values())

train_p, temp_p, train_l, temp_l = train_test_split(
    all_paths, all_labels,
    test_size=0.2, random_state=SEED,
    stratify=all_labels if can_strat else None)

temp_counts = Counter(temp_l)
can_strat_temp = can_strat and all(c >= 2 for c in temp_counts.values())

val_p, test_p, val_l, test_l = train_test_split(
    temp_p, temp_l,
    test_size=0.5, random_state=SEED,
    stratify=temp_l if can_strat_temp else None)

print(f'Train: {len(train_p)}  Val: {len(val_p)}  Test: {len(test_p)}')
assert len(set(train_p) & set(val_p)) == 0, 'Leakage: train ∩ val'
assert len(set(train_p) & set(test_p)) == 0, 'Leakage: train ∩ test'
assert len(set(val_p) & set(test_p)) == 0, 'Leakage: val ∩ test'

In [ ]:
# Build TF datasets — uses decode_image (handles JPEG *and* PNG)

def make_dataset(paths, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((list(paths), list(labels)))
    if shuffle:
        ds = ds.shuffle(len(paths), seed=SEED)

    def process(fpath, label):
        raw = tf.io.read_file(fpath)
        img = tf.image.decode_image(raw, channels=3, expand_animations=False)
        img.set_shape([None, None, 3])
        img = tf.image.resize(img, IMG_SIZE)
        return img, tf.one_hot(label, NUM_CLASSES)

    return (ds
            .map(process, num_parallel_calls=tf.data.AUTOTUNE)
            .batch(BATCH_SIZE)
            .prefetch(tf.data.AUTOTUNE))

train_ds = make_dataset(train_p, train_l, shuffle=True)
val_ds   = make_dataset(val_p,   val_l)
test_ds  = make_dataset(test_p,  test_l)

print('Datasets ready.')

## 6 · Model Definition (MobileNetV2)

In [ ]:
def build_model(num_classes):
    base = tf.keras.applications.MobileNetV2(
        input_shape=IMG_SIZE + (3,),
        include_top=False,
        weights='imagenet')
    base.trainable = False

    aug = tf.keras.Sequential([
        tf.keras.layers.RandomFlip('horizontal'),
        tf.keras.layers.RandomRotation(0.2),
        tf.keras.layers.RandomZoom(0.1),
    ], name='augmentation')

    inp = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = aug(inp)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    out = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

    m = tf.keras.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
    return m

model = build_model(NUM_CLASSES)
model.summary()

## 7 · Train the Model

Two checkpoints are saved to Drive:

| File | Purpose |
|---|---|
| `best_model.h5` | Best validation loss — used for evaluation & Streamlit |
| `latest_ckpt.h5` | Every epoch — used to **resume** after a disconnect |

A small JSON file (`training_progress.json`) records the last completed
epoch.  If Colab disconnects, simply re-run from Cell 1: the code will
detect the latest checkpoint, load weights + optimizer state, and continue
from the correct epoch.

In [ ]:
BEST_PATH     = os.path.join(ARTIFACTS_DIR, 'best_model.h5')
LATEST_PATH   = os.path.join(ARTIFACTS_DIR, 'latest_ckpt.h5')
PROGRESS_PATH = os.path.join(ARTIFACTS_DIR, 'training_progress.json')
LOG_PATH      = os.path.join(ARTIFACTS_DIR, 'training_log.csv')

EPOCHS = 15
initial_epoch = 0

# --- Resume logic ---
if os.path.exists(LATEST_PATH):
    try:
        model = tf.keras.models.load_model(LATEST_PATH)
        if os.path.exists(PROGRESS_PATH):
            with open(PROGRESS_PATH) as f:
                initial_epoch = json.load(f).get('completed_epoch', 0)
        print(f'✅ Resumed from latest checkpoint (epoch {initial_epoch})')
    except Exception as e:
        print(f'⚠️  Could not load checkpoint: {e}. Training from scratch.')
        initial_epoch = 0
else:
    print('Starting training from scratch.')
    if os.path.exists(LOG_PATH):
        os.remove(LOG_PATH)

class LatestCheckpoint(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        self.model.save(LATEST_PATH)
        with open(PROGRESS_PATH, 'w') as f:
            json.dump({'completed_epoch': epoch + 1}, f)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        BEST_PATH, save_best_only=True,
        monitor='val_loss', mode='min'),
    LatestCheckpoint(),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=3,
        restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.2,
        patience=2, min_lr=1e-6),
    tf.keras.callbacks.CSVLogger(LOG_PATH, append=True),
]

if initial_epoch < EPOCHS:
    history = model.fit(
        train_ds, validation_data=val_ds,
        epochs=EPOCHS, initial_epoch=initial_epoch,
        callbacks=callbacks)
    print('Training complete.')
else:
    history = None
    print(f'Already completed {initial_epoch} epochs — nothing to do.')

## 8 · Evaluate on Test Set & Save Artifacts

In [ ]:
# Load the best model for evaluation
if os.path.exists(BEST_PATH):
    eval_model = tf.keras.models.load_model(BEST_PATH)
    print('Loaded best_model.h5 for evaluation.')
else:
    eval_model = model
    print('No best checkpoint found — evaluating the current model.')

loss, accuracy = eval_model.evaluate(test_ds)
print(f'Test Loss:     {loss:.4f}')
print(f'Test Accuracy: {accuracy:.4f}')

y_true, y_pred = [], []
for imgs, lbls in test_ds:
    preds = eval_model.predict(imgs, verbose=0)
    y_true.extend(np.argmax(lbls.numpy(), axis=-1))
    y_pred.extend(np.argmax(preds, axis=-1))

# Classification report
all_labels = list(range(NUM_CLASSES))
report_text = classification_report(
    y_true, y_pred,
    labels=all_labels, target_names=CLASS_NAMES,
    zero_division=0)
report_dict = classification_report(
    y_true, y_pred,
    labels=all_labels, target_names=CLASS_NAMES,
    output_dict=True, zero_division=0)
print(report_text)

# Save class mapping
mapping_path = os.path.join(ARTIFACTS_DIR, 'class_mapping.json')
with open(mapping_path, 'w') as f:
    json.dump({'classes': CLASS_NAMES}, f)
print(f'Class mapping saved to {mapping_path}')

# Save evaluation results
eval_path = os.path.join(ARTIFACTS_DIR, 'evaluation_results.json')
with open(eval_path, 'w') as f:
    json.dump({'test_loss': float(loss), 'test_accuracy': float(accuracy),
               'classification_report': report_dict}, f, indent=2)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=all_labels)
fig_size = max(10, NUM_CLASSES * 0.55)
plt.figure(figsize=(fig_size, fig_size))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACTS_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

# Training history plots
if history is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(history.history['accuracy'], label='Train')
    ax1.plot(history.history['val_accuracy'], label='Val')
    ax1.set_title('Accuracy'); ax1.legend()
    ax2.plot(history.history['loss'], label='Train')
    ax2.plot(history.history['val_loss'], label='Val')
    ax2.set_title('Loss'); ax2.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(ARTIFACTS_DIR, 'training_history.png'), dpi=150)
    plt.show()

print(f'\nAll artifacts saved to: {ARTIFACTS_DIR}')
print('Files you need locally for the Streamlit app:')
print(f'  1. {BEST_PATH}')
print(f'  2. {mapping_path}')

## 9 · How to Resume After a Disconnect

1. Re-open this notebook in Colab.
2. Go to **Runtime → Change runtime type** and select **GPU**.
3. Run **all cells from the top** (Cell 1 onwards).
4. The dataset will not be re-downloaded (ZIP exists on Drive).
5. Training will automatically resume from `latest_ckpt.h5` at the
   correct epoch, with optimizer state and learning rate preserved.
6. The best model (`best_model.h5`) is never overwritten by the resume
   checkpoint — it is only updated when validation loss improves.